# GOV-01 Controlled Experiment: Class-Weighted CNN

This notebook repeats the compact CNN baseline with **one intended change**: class weights.

The smaller `Normal` class receives a higher training penalty when it is misclassified. The data split, seed, image size, model structure, augmentation, optimizer, epochs, and validation metric remain the same.

**Do not load or evaluate the protected test split.**

## Before running in Colab

Use the same preparation route as before: clone the latest repository, upload `Dataset.zip`, extract it into `data/raw`, and run `src/build_clean_split.py` with seed `42`. Then run this notebook from top to bottom.

The baseline evidence is already in the repository. This notebook will add one new row for `cnn_class_weighted_v2` to `reports/experiment_record.csv`.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['Normal', 'Pothole']
DATA_DIR = Path('data/processed/clean_split')
REPORTS_DIR = Path('reports')
RUN_NAME = 'cnn_class_weighted_v2'

tf.keras.utils.set_random_seed(SEED)
REPORTS_DIR.mkdir(exist_ok=True)

for split_name in ['train', 'validation', 'test']:
    if not (DATA_DIR / split_name).is_dir():
        raise FileNotFoundError(f'Missing folder: {DATA_DIR / split_name}')

print('Clean split found:', DATA_DIR.resolve())
print('Protected test folder exists but will not be loaded.')

In [ ]:
def load_split(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split_name, labels='inferred', label_mode='binary',
        class_names=CLASS_NAMES, color_mode='rgb', image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED if shuffle else None,
    )

train_ds = load_split('train', shuffle=True)
validation_ds = load_split('validation', shuffle=False)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
], name='training_augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda images, labels: (data_augmentation(images, training=True), labels), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

print('Training and validation datasets are ready.')

In [ ]:
# The only intended experiment change: class weights.
# Normal has 236 training images; Pothole has 624.
train_counts = {0: 236, 1: 624}
train_total = sum(train_counts.values())
class_weight = {label: train_total / (2 * count) for label, count in train_counts.items()}

print('Class names:', CLASS_NAMES)
print('Class weights:', class_weight)
print('Normal receives a larger penalty because it has fewer training images.')

In [ ]:
# Same CNN structure as cnn_unweighted_v1.
# The class_weight argument in model.fit is the single planned difference.
tf.keras.utils.set_random_seed(SEED)

weighted_cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMAGE_SIZE + (3,)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation='sigmoid', name='pothole_probability'),
], name=RUN_NAME)

weighted_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True,
)
weighted_cnn.summary()

In [ ]:
start_time = time.time()
history = weighted_cnn.fit(
    train_ds, validation_data=validation_ds, epochs=20,
    class_weight=class_weight, callbacks=[early_stopping], verbose=1,
)
training_seconds = time.time() - start_time
print(f'Training time: {training_seconds:.1f} seconds')

In [ ]:
# Validation evaluation only. The protected test split is not loaded.
y_validation = np.concatenate([labels.numpy().ravel() for _, labels in validation_ds])
scores = weighted_cnn.predict(validation_ds).ravel()
predictions = (scores >= 0.50).astype(int)

weighted_metrics = {
    'accuracy': float(accuracy_score(y_validation, predictions)),
    'macro_f1': float(f1_score(y_validation, predictions, average='macro', zero_division=0)),
    'pothole_precision': float(precision_score(y_validation, predictions, pos_label=1, zero_division=0)),
    'pothole_recall': float(recall_score(y_validation, predictions, pos_label=1, zero_division=0)),
    'normal_recall': float(recall_score(y_validation, predictions, pos_label=0, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_validation, scores)),
}

new_record = {
    'run_name': RUN_NAME,
    'hypothesis': 'Class weights improve Normal recall and Macro F1 without changing the CNN architecture.',
    'changed_factor': 'class_weight: none to balanced weights',
    'class_weight': f'Normal={class_weight[0]:.3f}; Pothole={class_weight[1]:.3f}',
    'training_seconds': round(training_seconds, 1),
    **weighted_metrics,
}

record_path = REPORTS_DIR / 'experiment_record.csv'
previous_records = pd.read_csv(record_path) if record_path.exists() else pd.DataFrame()
previous_records = previous_records[previous_records['run_name'] != RUN_NAME] if not previous_records.empty else previous_records
comparison = pd.concat([previous_records, pd.DataFrame([new_record])], ignore_index=True)
comparison.to_csv(record_path, index=False)
display(comparison)

print(classification_report(y_validation, predictions, target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_validation, predictions, display_labels=CLASS_NAMES)
plt.title('Validation confusion matrix: cnn_class_weighted_v2')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'cnn_class_weighted_v2_validation_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
history_frame = pd.DataFrame(history.history)
history_frame[['loss', 'val_loss']].plot(title='Class-weighted CNN training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'cnn_class_weighted_v2_learning_curve.png', dpi=150)
plt.show()

print('Saved new validation evidence in reports/.')
print('Do not use the test split yet.')